# MADDPG Baseline — Results

**Task:** `simple_tag_v3` — 3 chasers vs 1 evader  
**Algorithm:** MADDPG (centralized training, decentralized execution)  

Figures are saved to the run directory alongside `metrics.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.frameon": False,
})

COLORS = {"adversary": "#2196F3", "agent": "#F44336"}
SMOOTH_WINDOW = 10  # rolling mean window for noisy train metrics
fmt_ksteps = mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k")

In [ ]:
# Finds the most recent run that has a complete metrics.csv (with eval columns).
# Override by setting RUN_DIR explicitly, e.g.:
# RUN_DIR = Path("runs/20260319_210851")

EVAL_COLS = ["eval_ep_return_adversary", "eval_ep_return_agent"]

def find_latest_run(runs_root: Path) -> tuple[Path, pd.DataFrame]:
    for run_dir in sorted(runs_root.iterdir(), reverse=True):
        csv = run_dir / "metrics.csv"
        if not csv.exists():
            continue
        df = pd.read_csv(csv)
        if all(c in df.columns for c in EVAL_COLS):
            return run_dir, df
    raise FileNotFoundError(f"No complete run with eval columns found in {runs_root}")

RUN_DIR, df = find_latest_run(Path("runs"))
print(f"Run: {RUN_DIR}  |  {len(df)} iterations")

eval_df = df.dropna(subset=EVAL_COLS).copy()
print(f"Eval checkpoints: {len(eval_df)}")

def smooth(s):
    return s.rolling(SMOOTH_WINDOW, min_periods=1, center=True).mean()

df.tail(3)

## Figure 1 — Eval Learning Curve
Episode return under the deterministic policy, averaged over 10 episodes every 10 training iterations.  
This is the primary comparable metric (matches the MADDPG paper reporting convention).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for group, label in [("adversary", "Adversary (chasers)"), ("agent", "Agent (evader)")]:
    ax.plot(
        eval_df["total_frames"],
        eval_df[f"eval_ep_return_{group}"],
        marker="o", markersize=4,
        color=COLORS[group], label=label,
    )

ax.set_xlabel("Environment Steps")
ax.set_ylabel("Episode Return")
ax.set_title("Eval Episode Return (Deterministic Policy)")
ax.xaxis.set_major_formatter(fmt_ksteps)
ax.legend()

plt.tight_layout()
plt.savefig(RUN_DIR / "fig_eval_returns.png", bbox_inches="tight")
plt.show()

## Figure 2 — Train Episode Return
Per-iteration return computed from the collector batch (includes exploration noise).  
Faint line = raw, solid line = rolling mean.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for group, label in [("adversary", "Adversary (chasers)"), ("agent", "Agent (evader)")]:
    col = f"return_{group}"
    ax.plot(df["total_frames"], df[col], alpha=0.15, color=COLORS[group])
    ax.plot(df["total_frames"], smooth(df[col]), color=COLORS[group], label=label)

ax.set_xlabel("Environment Steps")
ax.set_ylabel("Episode Return")
ax.set_title("Train Episode Return (with smoothing)")
ax.xaxis.set_major_formatter(fmt_ksteps)
ax.legend()

plt.tight_layout()
plt.savefig(RUN_DIR / "fig_train_returns.png", bbox_inches="tight")
plt.show()

## Figure 3 — Training Diagnostics
Heuristic metrics computed from raw collector observations each iteration.  
Useful for understanding *how* the agents are learning to coordinate.

In [ ]:
diagnostics = [
    ("capture_rate",    "Capture Rate",         "Fraction of Episodes"),
    ("time_to_capture", "Time to First Capture", "Normalized Step (0=start, 1=end)"),
    ("collision_rate",  "Predator Collisions",   "Collisions / Step"),
    ("coverage_eff",    "Coverage Efficiency",   "Mean Pairwise Distance"),
]

# Only plot diagnostics that are present in this run's CSV
available = [(col, title, ylabel) for col, title, ylabel in diagnostics if col in df.columns]
if not available:
    print("No diagnostic metrics in this run's CSV — skipping Figure 3.")
else:
    fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4))
    if len(available) == 1:
        axes = [axes]

    for ax, (col, title, ylabel) in zip(axes, available):
        series = df[col].dropna()
        frames = df.loc[series.index, "total_frames"]
        ax.plot(frames, series, alpha=0.15, color="#555")
        ax.plot(frames, smooth(series), color="#222")
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.set_xlabel("Environment Steps")
        ax.xaxis.set_major_formatter(fmt_ksteps)

    plt.suptitle("Training Diagnostics", fontsize=13)
    plt.tight_layout()
    plt.savefig(RUN_DIR / "fig_diagnostics.png", bbox_inches="tight")
    plt.show()